# Python Mastery — Module 9
# Deep Learning for Computer Vision

A practical beginner-to-advanced introduction to image data, convolutional neural networks (CNNs), convolution filters, padding, stride, pooling, preprocessing, augmentation, transfer learning, and image-classification evaluation.

## Learning outcomes
- Explain how images are represented as tensors.
- Understand convolution kernels, feature maps, padding, stride, and pooling.
- Build a small CNN and train it on a real image dataset when available.
- Preprocess and augment images without leaking validation/test data.
- Use transfer learning with a pretrained CNN when internet/model weights are available.
- Evaluate classification results and diagnose overfitting.
- Understand practical constraints such as class imbalance, compute, reproducibility, and data quality.

**Important:** The examples use public toy datasets or synthetic images. A model's success on a toy dataset does not demonstrate suitability for real-world medical, safety, or business decisions.

## 1. How a computer represents an image

A grayscale image is commonly represented as a 2D array: height × width. A color image often has three channels: red, green, and blue (RGB).

A batch of color images is often shaped as:
- channels-last: `(batch, height, width, channels)` — common in TensorFlow/Keras;
- channels-first: `(batch, channels, height, width)` — common in PyTorch.

Pixel values are commonly integers from 0 to 255. Neural networks often work better when values are converted to floating point and scaled, for example to 0–1. Pretrained models may require a model-specific preprocessing function instead.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Make a small illustrative RGB image using arrays, no downloads required.
toy_image = np.zeros((64, 64, 3), dtype=np.uint8)
toy_image[8:56, 8:56, 0] = 220
toy_image[20:44, 20:44, 1] = 180
toy_image[28:36, 28:36, 2] = 255

print("Image shape (height, width, channels):", toy_image.shape)
print("Pixel value range:", toy_image.min(), "to", toy_image.max())

plt.figure(figsize=(4, 4))
plt.imshow(toy_image)
plt.title("Illustrative RGB image")
plt.axis("off")
plt.show()

## 2. What a convolutional neural network (CNN) does

A CNN learns spatial patterns in images. Early layers often learn simple edges or textures; deeper layers can combine these into more complex patterns. This is a useful intuition, not a guarantee that each filter has a neat human interpretation.

A common image classifier pipeline is:

**Image → convolution + activation → pooling/downsampling → repeated feature extraction → classifier → class scores**

CNNs use local receptive fields and shared weights. A filter is reused across the image, which usually means fewer parameters than connecting every pixel directly to every hidden neuron.

## 3. Convolution and filters

A convolution filter (kernel) is a small matrix of learned weights that slides across an image. At each location, it combines nearby pixel values to create one value in a feature map.

- **Kernel size:** spatial size of the filter, such as 3×3.
- **Feature map:** output produced by applying a filter.
- **Channels:** color channels in the input and learned feature channels in the output.
- **Learned filters:** weights are learned from data through backpropagation.

The following toy example uses a fixed edge-like filter to illustrate the operation. In a CNN, filters are typically learned rather than manually selected.

In [ ]:
# A simple 2D convolution example implemented with NumPy loops for clarity.
gray = np.zeros((8, 8), dtype=float)
gray[2:6, 3:5] = 1.0
kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
], dtype=float)

def simple_valid_convolution(image, kernel):
    h, w = image.shape
    kh, kw = kernel.shape
    output = np.zeros((h - kh + 1, w - kw + 1), dtype=float)
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            output[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return output

feature_map = simple_valid_convolution(gray, kernel)
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(gray, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Input image")
axes[1].imshow(kernel, cmap="coolwarm")
axes[1].set_title("Fixed filter")
axes[2].imshow(feature_map, cmap="coolwarm")
axes[2].set_title("Feature map")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Padding, stride, and output dimensions

- **Padding:** adds border values (often zeros) around the input. `valid` means no padding; `same` generally aims to preserve spatial dimensions when stride is 1.
- **Stride:** number of pixels the filter moves at each step. A larger stride reduces output resolution.
- **Dilation:** spaces kernel elements apart and increases the receptive field without increasing the number of kernel weights.

For one spatial dimension, the output size is:

\[
\text{output} = \left\lfloor \frac{N + 2P - D(K-1) - 1}{S} \right\rfloor + 1
\]

where `N` is input size, `P` padding, `D` dilation, `K` kernel size, and `S` stride.

In [ ]:
def conv_output_size(n, kernel_size, padding=0, stride=1, dilation=1):
    return ((n + 2 * padding - dilation * (kernel_size - 1) - 1) // stride) + 1

examples = []
for name, padding, stride in [
    ("valid, stride 1", 0, 1),
    ("same-like padding, stride 1", 1, 1),
    ("valid, stride 2", 0, 2),
]:
    examples.append({
        "setting": name,
        "input_size": 32,
        "kernel": 3,
        "padding": padding,
        "stride": stride,
        "output_size": conv_output_size(32, 3, padding, stride),
    })
import pandas as pd
pd.DataFrame(examples)

## 5. Pooling and downsampling

Pooling reduces spatial resolution:
- **Max pooling:** keeps the largest value in each local window.
- **Average pooling:** computes the average within each window.
- **Global average pooling:** averages each feature map to one value, often reducing parameters before classification.

Pooling can reduce computation and make later layers less sensitive to small shifts. However, downsampling discards detail. Modern architectures may use strided convolutions or other downsampling methods instead of conventional pooling.

In [ ]:
# Demonstrate 2x2 max pooling on a tiny feature map.
feature = np.array([
    [1, 3, 2, 0],
    [4, 6, 1, 2],
    [0, 2, 8, 5],
    [1, 3, 7, 4],
])
def max_pool_2x2(arr):
    h, w = arr.shape
    result = np.zeros((h // 2, w // 2), dtype=arr.dtype)
    for i in range(0, h, 2):
        for j in range(0, w, 2):
            result[i//2, j//2] = np.max(arr[i:i+2, j:j+2])
    return result

print("Input feature map:\n", feature)
print("After 2x2 max pooling:\n", max_pool_2x2(feature))

## 6. A typical CNN architecture

A simple CNN for image classification often includes:
1. convolution layer(s);
2. activation such as ReLU;
3. pooling or another downsampling step;
4. more convolution blocks;
5. flattening or global average pooling;
6. dense classification layer.

A common mistake is flattening a very large feature map too early, creating a huge number of parameters. Global average pooling or smaller feature maps can reduce the parameter count.

The output layer and loss must match the task. For mutually exclusive classes, the model often returns class logits and uses a cross-entropy loss.

## 7. Image preprocessing

Typical preprocessing steps:
- read image files and decode the format;
- ensure consistent channel ordering and color mode;
- resize or crop to the model's expected dimensions;
- convert to floating point;
- normalize pixels or apply the pretrained model's required preprocessing;
- encode labels consistently;
- inspect corrupt images, duplicates, and class imbalance.

**Split before augmentation.** Augmented versions of the same source image must not appear in both training and validation/test sets. For people, patients, objects, videos, or near-duplicate images, split at the entity/source level when needed.

## 8. Data augmentation

Augmentation creates plausible variations of training images, such as:
- horizontal flips when the label remains valid;
- small rotations or translations;
- random crops and zooms;
- brightness/contrast changes;
- random erasing or cutout.

Augmentation can improve robustness but must respect the domain. A transformation that changes the meaning of an image or label is harmful. For example, a horizontal flip may be invalid for images where left/right orientation is clinically meaningful.

Use augmentation only on training data; validation and test preprocessing should be deterministic.

## 9. Practical CNN using scikit-learn's digits dataset

The handwritten-digits dataset is small, grayscale, and available locally through scikit-learn. It is useful for a first CNN demonstration without downloading a large image dataset.

We will:
- inspect the 8×8 images;
- split into train/test sets;
- scale pixel values;
- build a small CNN using PyTorch if installed;
- evaluate accuracy and a confusion matrix.

This dataset is a learning exercise, not a benchmark for real image products.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

digits = load_digits()
X_digits = digits.images.astype("float32") / 16.0  # pixels are 0..16
y_digits = digits.target.astype("int64")

X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_digits, y_digits, test_size=0.2, stratify=y_digits, random_state=42
)
print("Train image tensor:", X_train_img.shape)
print("Test image tensor:", X_test_img.shape)

fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for ax, image, label in zip(axes, X_train_img[:8], y_train_img[:8]):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Build and train a small CNN with PyTorch

This section is optional if PyTorch is unavailable. For the digits dataset, we use a small network to keep training quick.

PyTorch CNN inputs use channels-first shape `(batch, channels, height, width)`. The 8×8 digits therefore need a singleton channel dimension.

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    print("PyTorch version:", torch.__version__)
except ImportError:
    print("PyTorch is not installed; you can skip the CNN training cells.")

In [ ]:
if "torch" in globals():
    Xtr = torch.tensor(X_train_img[:, None, :, :], dtype=torch.float32)
    Xte = torch.tensor(X_test_img[:, None, :, :], dtype=torch.float32)
    ytr = torch.tensor(y_train_img, dtype=torch.long)
    yte = torch.tensor(y_test_img, dtype=torch.long)

    train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

    class SmallCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(1, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.AdaptiveAvgPool2d((2, 2)),
            )
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(32 * 2 * 2, 64),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(64, 10),
            )

        def forward(self, x):
            return self.classifier(self.features(x))

    cnn = SmallCNN()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)

    for epoch in range(12):
        cnn.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = cnn(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(xb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += len(xb)
        if (epoch + 1) % 3 == 0:
            print(f"Epoch {epoch+1:02d}: loss={running_loss/total:.4f}, train accuracy={correct/total:.3f}")
else:
    print("CNN training skipped because PyTorch is unavailable.")

## 11. Evaluate the CNN

Accuracy is intuitive, but it can hide weak performance on rare classes. Use the confusion matrix and per-class precision/recall, especially when class frequencies differ or errors have different costs.

The test set should be used for final evaluation only. For serious experiments, keep a validation set for choosing epochs and hyperparameters rather than repeatedly checking test performance.

In [ ]:
if "torch" in globals():
    cnn.eval()
    with torch.no_grad():
        test_logits = cnn(Xte)
        y_pred_digits = test_logits.argmax(dim=1).cpu().numpy()
    print("Test accuracy:", round(accuracy_score(y_test_img, y_pred_digits), 4))
    print(classification_report(y_test_img, y_pred_digits, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test_img, y_pred_digits, cmap=None)
    plt.title("CNN confusion matrix")
    plt.show()
else:
    print("Evaluation skipped because PyTorch is unavailable.")

## 12. Overfitting in computer vision

Image models can memorize training examples, backgrounds, camera artifacts, or repeated sources rather than learning the intended object.

Signs:
- training performance improves while validation performance worsens;
- large performance gaps between training and validation;
- strong performance on one source/device but poor performance elsewhere.

Mitigations:
- stronger but valid augmentation;
- weight decay and dropout;
- early stopping;
- a smaller model;
- more diverse data;
- deduplication and source-level splits;
- validation that resembles deployment conditions.

## 13. Transfer learning

Transfer learning starts from a model pretrained on a large dataset and adapts it to a new task. It is often effective when the target dataset is not large enough to train a deep CNN from scratch.

Common approaches:
1. **Feature extraction:** freeze the pretrained backbone and train a new classifier head.
2. **Fine-tuning:** unfreeze some or all backbone layers and continue training with a small learning rate.

Practical cautions:
- use the preprocessing expected by the pretrained weights;
- match the model's input size and channel order;
- replace the final classification head;
- freeze/unfreeze intentionally;
- use a lower learning rate when fine-tuning;
- evaluate on a truly separate test set;
- check whether the source dataset's features transfer to the target domain.

The next code cell shows a torchvision example. It may require downloading pretrained weights; if the environment is offline, use `weights=None` or skip it.

In [ ]:
# Optional transfer-learning example. It requires torchvision and may need internet
# access to download pretrained weights the first time.
try:
    import torchvision
    from torchvision.models import resnet18, ResNet18_Weights
    print("torchvision version:", torchvision.__version__)
except Exception as exc:
    torchvision = None
    print("torchvision unavailable:", type(exc).__name__, str(exc)[:160])

In [ ]:
if "torch" in globals() and torchvision is not None:
    try:
        weights = ResNet18_Weights.DEFAULT
        transfer_model = resnet18(weights=weights)
        # Freeze pretrained feature extractor.
        for param in transfer_model.parameters():
            param.requires_grad = False
        # Replace the final classifier for a hypothetical 3-class task.
        in_features = transfer_model.fc.in_features
        transfer_model.fc = nn.Linear(in_features, 3)
        # Only the new classification head is trainable.
        trainable = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in transfer_model.parameters())
        print(f"Trainable parameters: {trainable:,} / {total:,}")
        print("Expected input: RGB images resized and normalized with the selected weights' transforms.")
    except Exception as exc:
        print("Pretrained weights/model could not be loaded:", type(exc).__name__, str(exc)[:200])
        print("You can still learn transfer learning by using weights=None or an offline model checkpoint.")
else:
    print("Transfer learning demo skipped because PyTorch/torchvision is unavailable.")

## 14. Data loaders and compute efficiency

For larger image datasets, avoid loading every full-resolution image into memory at once. Use a data pipeline or `Dataset`/`DataLoader` that decodes images and yields batches.

Useful practices:
- choose a batch size that fits memory;
- shuffle training data, but keep validation/test ordering deterministic when practical;
- use pinned memory and workers thoughtfully when using a GPU;
- resize to the smallest resolution that preserves task-relevant information;
- use mixed precision only when supported and validated;
- monitor data-loading time as well as GPU utilization;
- save checkpoints and training metadata.

## 15. Common mistakes

1. Using the wrong tensor channel order.
2. Forgetting to scale pixels or applying the wrong pretrained-model normalization.
3. Using a classification loss that does not match label format and output logits.
4. Augmenting validation or test images randomly.
5. Allowing near-duplicate images or images from the same subject/source to cross splits.
6. Fine-tuning a pretrained network with an overly high learning rate.
7. Choosing epochs by looking at test results.
8. Reporting accuracy alone on an imbalanced dataset.
9. Assuming transfer learning always helps across domains.
10. Interpreting model performance as proof that the model learned the intended visual concept.

## 16. Exercises

Try these before reading the suggested answers.

1. Why do CNNs use filters shared across an image?
2. What is a feature map?
3. How do padding and stride affect output dimensions?
4. What does max pooling do?
5. Why should image augmentation be applied only to training data?
6. Why might splitting images by patient, person, or video source be necessary?
7. What is the difference between feature extraction and fine-tuning?
8. Why must pretrained-model preprocessing match the selected weights?
9. Why can accuracy be misleading for rare classes?
10. Give one reason a CNN might perform well in training but poorly in deployment.

### Suggested answers

1. Weight sharing reduces parameter count and lets a learned pattern be detected at different spatial locations.
2. A feature map is the output produced by applying a filter across the input.
3. Padding changes the effective border size; larger stride takes fewer positions and usually reduces spatial resolution.
4. It retains the maximum value in each local window, reducing resolution while preserving strong activations.
5. Randomly augmenting validation/test data makes evaluation inconsistent and can make it unlike the intended deployment pipeline.
6. Related images can be near-duplicates, so the model may memorize source-specific information and produce overoptimistic scores.
7. Feature extraction freezes the pretrained backbone and trains a new head; fine-tuning updates some or all pretrained layers as well.
8. The model expects a particular input scale, normalization, and sometimes image size; mismatches can harm predictions.
9. A model can predict the majority class most of the time while missing the rare class.
10. Distribution shift, leakage in the training split, background shortcuts, or insufficient source diversity can cause failure.

## 17. Capstone — build an image classifier

### Goal
Build a reproducible image classification workflow using a public image dataset appropriate to your compute budget.

### Tasks
1. Define classes and the intended prediction use.
2. Inspect examples, dimensions, label counts, and corrupt/duplicate files.
3. Split at the independent source level before augmentation.
4. Build a simple baseline.
5. Implement a CNN with a suitable loss and optimizer.
6. Add valid training-only augmentation.
7. Use validation metrics for early stopping and tuning.
8. Evaluate once on a held-out test set with a confusion matrix and per-class metrics.
9. Compare training from scratch with transfer learning when practical.
10. Document limitations, data coverage, possible shortcut learning, and monitoring.

### Model card outline
- Intended use and prohibited use
- Dataset provenance and license
- Classes, sample counts, and split strategy
- Preprocessing and augmentation
- Architecture and pretrained weights
- Metrics by class and important subgroups
- Known limitations and failure modes
- Deployment constraints and monitoring plan

## 18. Readiness checklist

- [ ] Explain image tensor dimensions and channel order.
- [ ] Explain convolution, filters, feature maps, padding, and stride.
- [ ] Explain pooling and downsampling.
- [ ] Preprocess images consistently.
- [ ] Design label-preserving training augmentation.
- [ ] Build and train a small CNN.
- [ ] Evaluate using a confusion matrix and per-class metrics.
- [ ] Diagnose overfitting and data leakage in image datasets.
- [ ] Explain feature extraction versus fine-tuning.
- [ ] Match pretrained weights to the correct input preprocessing.
- [ ] Describe limitations and deployment risks.

### Next module
**Module 10 — Natural Language Processing and Transformers:** text preprocessing, embeddings, attention, transformer blocks, tokenization, pretrained language models, fine-tuning, and evaluation.